# MeshAPI Gateway Basics — Mistral + Groq

This notebook is a teaching intro to **MeshAPI** (`https://api.meshapi.ai`), an AI model gateway:
one API key, one OpenAI-shaped API, many providers behind it addressed as `provider/model` strings.

We'll route to two real providers through the *same* gateway client:
- **Mistral** (`mistral/...`)
- **Groq** (`groq/...`, known for very fast inference)

What this notebook covers:
1. Client setup + auth
2. Discovering live model IDs per provider (don't hardcode guesses)
3. A basic chat completion, run against both providers with zero code changes
4. Streaming
5. `compare` — run the same prompt across multiple providers at once
6. Error handling

The companion notebook `02_rag_multiagent.ipynb` builds a RAG + multi-agent workflow on top of this.

## 1. Install

In [ ]:
%pip install -q meshapi

## 2. Config & client

The SDK does **not** auto-read env vars — you pass `base_url` and `token` explicitly.
Get your `rsk_...` key from the MeshAPI dashboard.

In [ ]:
import os, json

from getpass import getpass



from meshapi import (

    MeshAPI, ChatCompletionParams, ChatMessage,

    MeshAPIError,

    CompareParams,

)



MESHAPI_BASE_URL = os.getenv("MESHAPI_BASE_URL", "https://api.meshapi.ai")

MESHAPI_TOKEN = os.getenv("MESHAPI_TOKEN") or getpass("MeshAPI token (rsk_...): ")



client = MeshAPI(base_url=MESHAPI_BASE_URL, token=MESHAPI_TOKEN)

print("Client ready.")

## 3. Helper: pretty-print any SDK response

SDK responses are Pydantic v2 models, so `model_dump_json()` always works even if you're
not sure of the exact field names yet — handy while exploring.

In [ ]:
def show(obj, indent=2):

    if hasattr(obj, "model_dump_json"):

        print(obj.model_dump_json(indent=indent))

    else:

        print(json.dumps(obj, indent=indent, default=str))

## 4. Discover live model IDs for Mistral and Groq

Model catalogs change over time, so instead of hardcoding a model slug we ask the gateway
what's actually available right now and pick from that.

In [ ]:
def list_model_ids(models_response):

    """Handles either a bare list or an object with a .data list, depending on SDK version."""

    items = getattr(models_response, "data", models_response)

    return [m.id for m in items]



mistral_ids = list_model_ids(client.models.list(provider="mistral"))

groq_ids = list_model_ids(client.models.list(provider="groq"))



print(f"Mistral models ({len(mistral_ids)}):")

for m in mistral_ids[:15]:

    print("  -", m)



print(f"\nGroq models ({len(groq_ids)}):")

for m in groq_ids[:15]:

    print("  -", m)

In [ ]:
# Pick a chat-capable model from each provider.

# Adjust the filters / index below if you want a specific model instead of the first match.

MISTRAL_MODEL = next((m for m in mistral_ids if "embed" not in m), mistral_ids[0])

GROQ_MODEL = next((m for m in groq_ids if "whisper" not in m), groq_ids[0])



print("Using MISTRAL_MODEL =", MISTRAL_MODEL)

print("Using GROQ_MODEL    =", GROQ_MODEL)

## 5. A basic chat completion

Same function, same code path — only the `model=` string changes between providers.
That's the entire pitch of a gateway.

In [ ]:
def ask(model, prompt, temperature=0.5, max_tokens=300):

    resp = client.chat.completions.create(

        ChatCompletionParams(

            model=model,

            messages=[ChatMessage(role="user", content=prompt)],

            temperature=temperature,

            max_tokens=max_tokens,

        )

    )

    return resp.choices[0].message.content



question = "In two sentences, what makes an AI model gateway useful for a startup?"



print("=== Mistral ===")

print(ask(MISTRAL_MODEL, question))



print("\n=== Groq ===")

print(ask(GROQ_MODEL, question))

## 6. Streaming

Groq is typically the fastest inference provider available — a nice provider to demo streaming with.

In [ ]:
print(f"Streaming from {GROQ_MODEL}:\n")



stream = client.chat.completions.stream(

    ChatCompletionParams(

        model=GROQ_MODEL,

        messages=[ChatMessage(role="user", content="Count from 1 to 5, one number per line, with a short fun fact about each number.")],

    )

)



for chunk in stream:

    if chunk.choices and chunk.choices[0].delta and chunk.choices[0].delta.content:

        print(chunk.choices[0].delta.content, end="", flush=True)

## 7. `compare` — same prompt, multiple providers at once

`client.compare` fans a single prompt out to several models in one call, optionally with an
LLM-generated synthesis step. Great for "which provider should I use for this task" demos.

In [ ]:
result = client.compare.create(

    CompareParams(

        models=[MISTRAL_MODEL, GROQ_MODEL],

        messages=[ChatMessage(role="user", content="What's the single biggest tradeoff of using an AI gateway vs calling providers directly?")],

    )

)



# Inspect the raw shape first — compare responses vary slightly by SDK version.

show(result)

### Streaming variant of compare

If you'd rather watch both models answer token-by-token, use `.stream()` instead of `.create()`.
Print the raw event once to see the exact shape your installed SDK version returns, then adapt.

In [ ]:
for event in client.compare.stream(

    CompareParams(

        models=[MISTRAL_MODEL, GROQ_MODEL],

        messages=[ChatMessage(role="user", content="Name one good use case for a fast/cheap model vs a slow/powerful one.")],

    )

):

    print(event)

## 8. Error handling

`MeshAPIError` carries structured fields — `status`, `error_code`, `request_id`, `retry_after_seconds` —
so you can branch on things like rate limits or spend caps instead of parsing strings.

In [ ]:
try:

    client.chat.completions.create(

        ChatCompletionParams(model="not-a-real-provider/not-a-real-model", messages=[ChatMessage(role="user", content="hi")])

    )

except MeshAPIError as e:

    print(f"[{e.status}] {e.error_code}: {e}")

    print("request_id:", e.request_id)

    if e.error_code == "rate_limit_exceeded":

        print("retry after:", e.retry_after_seconds, "seconds")

## 9. Cleanup

In [ ]:
client.close()

print("Done. Next: open 02_rag_multiagent.ipynb")